**Role in the paper.** Converts the cleaned station data (output of `metobs_cleaning_scripts/`) into the netCDF files read by the method notebooks: hourly series, daily min/mean/max and monthly means of the daily values, per network and subset. Set `city` and run once per city.

In [ ]:
import pandas as pd
from pathlib import Path
import xarray as xr
ROOT = Path.cwd().parent.parent
import numpy as np

In [ ]:
city = 'Zurich'

# to do , add coordinates in ds

In [ ]:
for network in [ 'meteo_stations', 'combined_network']: #['combined_network', 'city_network', 'meteo_stations']
  for subset in ['', '_gapfree']: #'_gapfree2018', '_total',,'_gapfree', '_total', '_gapfree',  '_total', '_gapfree_2021_2022','_gapfree_2020_2021'
    data_dir = ROOT / f"data/processed/observations/{city}/{network}"
    save_dir = ROOT / f"data/processed/observations/{city}/{network}/datasets/{subset}"
    save_dir.mkdir(parents=True, exist_ok=True)
    path = data_dir / f'{city}_{network}_cleaned{subset}.csv'
  
    if path.exists():
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        df.index = (df.index.tz_convert('UTC').tz_localize(None)) # Ensure timezone-aware timestamps are converted to naive UTC 

        #if a forward slash is present in the station name, replace it with an underscore
        df['name'] = df['name'].str.replace('/', '_')

        # Hourly means
        df_obs_hourly = (
            df.groupby('name')['value']
            .resample('h')
            .mean()
            .unstack(level=0)
        )

        ds_obs_hourly = df_obs_hourly.to_xarray().rename({'datetime': 'time'})
        ds_obs_hourly.to_netcdf(save_dir / f'{city}_{network}{subset}_hourly.nc')
            
        # Daily statistics
        daily_stats = {}
        monthly_stats = {}

        for stat in ['min', 'max', 'mean']:

            # Daily aggregation
            df_daily = (
                df.groupby('name')['value']
                .resample('D')
                .agg(stat)
                .unstack(level=0)
            )
            if stat != 'mean' and city=='Birmingham':
                for station in ['UKE00105909', 'UK000000000']:
                    df_meteo2 = pd.read_csv(str(ROOT / 'data/processed/observations/Birmingham/meteo_stations/Birmingham_meteo_data2_tmin_tmax.csv'), sep = ',')
                    df_meteo2['datetime'] = pd.to_datetime(df_meteo2['DATE'])
                    df_meteo2 = df_meteo2.set_index('datetime')
                    df_daily[station] =  df_meteo2[df_meteo2['STATION']==station][f'T{stat.upper()}']

            ds_daily = df_daily.to_xarray().rename({'datetime': 'time'})
            daily_stats[stat] = ds_daily
            ds_daily.to_netcdf(save_dir / f'{city}_{network}{subset}_daily_{stat}.nc')
            # Number of valid daily values per month
            valid_days = ds_daily.notnull().resample(time='MS').sum()

            # Number of days in each month
            days_in_month = xr.DataArray(
                ds_daily.time.to_index().to_period('M').to_timestamp().days_in_month,
                coords={'time': ds_daily.time},
                dims='time'
            ).resample(time='MS').max()

            # Monthly mean of daily values
            ds_monthly = ds_daily.resample(time='MS').mean()

            # Keep only months with > half the days present
            ds_monthly = ds_monthly.where(valid_days > (days_in_month / 2))

            monthly_stats[stat] = ds_monthly
            ds_monthly.to_netcdf(save_dir / f'{city}_{network}{subset}_monthlymean_daily{stat}.nc', mode="w")
    else:
        print(f"File {path} does not exist. Skipping.")
        